# 06 — Đo model

Ba bài đo. Hai bài đầu trả lời hai câu ngược nhau, phải đọc cạnh nhau mới có nghĩa; bài ba là cận trên trên chính dữ liệu nhà bạn, cùng loại giả định như bài CIC:

| Bài đo | Dữ liệu | Câu hỏi | Hỏng thì sao |
|---|---|---|---|
| **IoT Sentinel** | 31 thiết bị 2016, gần như không lớp nào trùng tập train | model có dám nói *không biết* với thiết bị lạ không | gán nhãn bừa cho thiết bị lạ = lỗi im lặng, không tầng nào ở trên bắt được |
| **CIC 2022** | chính tập train | thiết bị đã biết thì có nhận ra không | abstain hết = model an toàn nhưng vô dụng |
| **FIELD capture** | thiết bị nhà bạn (`Data/data_pcap/`), scenario `FIELD`, cũng nằm trong tập train | đường suy luận lúc chạy thật có tái hiện đúng các thiết bị đời thực này không | sai ở đây thường do windowing/nhãn field-capture, không phải do thiếu khái quát |

Notebook này nạp các định nghĩa từ `03_train_model.ipynb`; không import hay sinh file `.py`,
nên cái được đo đúng là cái sẽ chạy trên server.

Mở riêng để đo candidate đã chọn. Nếu một notebook khác gọi bằng `%run -i`,
`run_dir` của notebook gọi được dùng lại.

In [ ]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

In [1]:
# Nạp định nghĩa từ 03 mà không chạy train.
from pathlib import Path

_cwd = Path.cwd().resolve()
_ROOT = next((p for p in (_cwd, *_cwd.parents)
              if (p / "Code" / "03_train_model.ipynb").is_file()), None)
assert _ROOT is not None, f"Không thấy Code/03_train_model.ipynb quanh {_cwd}"
_CODE = _ROOT / "Code"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_CODE / "03_train_model.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import json
from collections import Counter, OrderedDict

import pandas as pd
from IPython.display import display

Configured for 88-core CPU
Gốc dự án: /home/ubuntu/sepcung/02.SDC


Đã nạp hàm SDC từ 03_train_model.ipynb


## Bộ đo IoT Sentinel

Bộ đo chạy trực tiếp trên định nghĩa trong `03_train_model.ipynb`.

In [2]:
import argparse
import sys
import json

from collections import Counter, OrderedDict
from pathlib import Path

import pandas as pd

SENTINEL_HOME = ROOT / "test_model"


# Bộ test nằm gọn trong thư mục này: pcap thô + cửa sổ đã chuẩn bị ở `data_test/`,
# model được chọn từ `Models/` hoặc truyền đường dẫn cụ thể.
SENTINEL_DIR = SENTINEL_HOME / "data_test"
SENTINEL_OUT = SENTINEL_DIR / "out"

# Head thứ ba tên `model` từ contract 2.2; bundle train trước đó gọi nó là `family`.
# Bảng nhãn thật của bộ test chỉ có một tên, nên tra theo cả hai để cùng một `coverage.csv`
# đo được cả model cũ lẫn model mới — khỏi phải giữ hai bản nhãn.
TRUTH_HEAD_ALIASES = {"model": ("model", "family"), "family": ("family", "model")}


def truth_for(meta, head):
    """Nhãn thật của `head` trong một dòng coverage, chấp nhận tên cột cũ."""
    for key in TRUTH_HEAD_ALIASES.get(head, (head,)):
        value = meta.get(key)
        if value:
            return str(value)
    return ""

# Verdict -> (nhãn in ra, có phải lỗi không). Tách riêng `BÁO NHẦM` khỏi `SAI` vì hai cái
# hỏng theo hai kiểu: `SAI` là nhầm giữa các lớp đã biết, `BÁO NHẦM` là gán nhãn cho thứ
# đáng lẽ phải báo lạ — cái sau koihông có tầng nào ở trên bắt được.
VERDICTS = OrderedDict([
    ("OK",            "nhận đúng thiết bị thuộc lớp đã biết"),
    ("OK MỞ",         "nhận đúng loại mới hoặc loại cha tương thích ngoài tập train"),
    ("SAI",           "trả lời sai nhãn giữa các lớp đã biết"),
    ("BỎ SÓT",        "lớp đã biết nhưng không dám trả lời"),
    ("PHÁT HIỆN LẠ",  "thiết bị lạ, đã báo lạ — đúng thiết kế"),
    ("BÁO NHẦM",      "thiết bị lạ nhưng vẫn gán nhãn — LỖI IM LẶNG"),
    ("CHƯA ĐỦ",       "chưa đủ cửa sổ để kết luận — không tính vào tỉ lệ"),
])


def judge(state, top1, truth, in_train, compatible=False):
    # `collecting` không phải một kết luận sai, nó là "chưa kết luận". Tính nó vào ô nào
    # trong bảng trên cũng làm hỏng cả hai tỉ lệ, nên tách riêng và trừ khỏi mẫu số.
    if state == "collecting":
        return "CHƯA ĐỦ"
    if in_train:
        if state != "identified":
            return "BỎ SÓT"
        return "OK" if top1 == truth else "SAI"
    if state == "identified" and compatible:
        return "OK MỞ"
    return "BÁO NHẦM" if state == "identified" else "PHÁT HIỆN LẠ"


# Ký hiệu so dự đoán với nhãn thật. Dùng ASCII để không phụ thuộc bảng mã console.
MARKS = OrderedDict([
    ("=",  "trùng đúng nhãn thật"),
    ("~",  "nhãn cha/loại tương thích với nhãn thật (theo catalog semantic)"),
    ("x",  "SAI — trả lời một nhãn khác nhãn thật"),
    ("-",  "không trả lời (có nhãn thật để so)"),
    ("?",  "có trả lời nhưng bộ test không có nhãn thật để so"),
    (" ",  "không trả lời và cũng không có nhãn thật"),
])


def match_mark(pred, gt, compatible=False):
    """So một dự đoán với nhãn thật -> một ký hiệu trong `MARKS`."""
    if not pred:
        return "-" if gt else " "
    if gt and pred == gt:
        return "="
    if compatible:
        return "~"
    return "x" if gt else "?"


def describe_policy(predictor):
    """Một dòng mô tả chính sách abstain đang dùng.

    Với `sdc-tiered-v2`, `predictor.thresholds` là bảng 12 ô (head × số nguồn); in
    nguyên dict ra thì không ai đọc nổi lúc so hai lần chạy.
    """
    if predictor.contract_format != TIERED_FORMAT:
        return "  ".join(f"{h}={predictor.thresholds[h]:.2f}" for h in predictor.heads)
    n_max = len(SOURCE_FLAGS)
    parts = []
    for head in predictor.heads:
        by_n = " ".join("{:.2f}".format(predictor.thresholds_by_source["{}|{}".format(head, n)])
                        for n in range(1, n_max + 1))
        parts.append(f"{head}[{by_n}] min={predictor.min_sources[head]}")
    return "  ".join(parts)


def load_windows(path):
    """windows.jsonl -> {device: [list bản ghi của từng cửa sổ]}, giữ thứ tự file."""
    by_device = OrderedDict()
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            w = json.loads(line)
            by_device.setdefault(w["device"], []).append(w)
    return by_device

In [3]:
def run_sentinel_test(argv=None):
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--run", type=Path, default=None,
                    help="thư mục model; mặc định dùng model ghim hoặc candidate duy nhất")
    ap.add_argument("--data", type=Path, default=SENTINEL_OUT / "windows.jsonl")
    ap.add_argument("--coverage", type=Path, default=SENTINEL_OUT / "coverage.csv")
    ap.add_argument("--out", type=Path, default=SENTINEL_OUT)
    ap.add_argument("--min-windows", type=int, default=3,
                    help="số cửa sổ tối thiểu trước khi DeviceTracker dám kết luận")
    ap.add_argument("--decide-ratio", type=float, default=0.7)
    ap.add_argument("--no-sticky", action="store_true",
                    help="không ghim DHCP/TLS giữa các cửa sổ (đo mức cửa sổ rời rạc)")
    ap.add_argument("--use-enrolled", action="store_true",
                    help="dùng cả bảng thu nạp Models/enrolled.json (mặc định: bỏ qua)")
    args = ap.parse_args(argv)

    for path in (args.data, args.coverage):
        if not path.exists():
            sys.exit(f"Không tìm thấy {path} — chạy cell chuẩn bị dữ liệu IoT Sentinel ở trên trước")

    # --- Nạp model --------------------------------------------------------------------
    run = select_run_dir(args.run)
    try:
        predictor = Predictor(run, enrolled=None if args.use_enrolled else {})
    except (ModelContractError, FileNotFoundError) as exc:
        sys.exit(f"Không nạp được model: {exc}\n"
                 f"Chỉ rõ thư mục model bằng --run <đường dẫn>")
    classes = {h: set(map(str, predictor.models[h].classes_)) for h in predictor.heads}
    print(f"Model  : {predictor.run_dir}  ({predictor.contract_format})")
    print(f"Ngưỡng : {describe_policy(predictor)}")
    if predictor.contract_format == TIERED_FORMAT:
        print(f"         ngưỡng xếp theo số nguồn bằng chứng 1..{len(SOURCE_FLAGS)}, "
              f"sàn L1 = {predictor.l1_floor:.2f}")
    print(f"Lớp    : " + "  ".join(f"{h}={len(classes[h])}" for h in predictor.heads))
    if predictor.semantic_type is not None:
        print(f"Catalog: {predictor.semantic_type.catalog_version}")

    truth_df = pd.read_csv(args.coverage).fillna("")
    truth = truth_df.set_index("device").to_dict("index")
    by_device = load_windows(args.data)
    if not by_device:
        sys.exit(f"{args.data} rỗng — chạy lại cell chuẩn bị dữ liệu IoT Sentinel")
    print(f"Dữ liệu: {len(by_device)} thiết bị, "
          f"{sum(len(v) for v in by_device.values())} cửa sổ\n")

    # --- Chạy -------------------------------------------------------------------------
    window_rows, device_rows = [], []

    for device, windows in by_device.items():
        meta = truth.get(device, {})
        tracker = DeviceTracker(predictor, min_windows=args.min_windows,
                                          decide_ratio=args.decide_ratio)
        for w in windows:
            out = tracker.observe(w["records"])
            if args.no_sticky:
                # Xoá bằng chứng đã ghim -> cửa sổ sau chỉ được dùng chính nó. Cách này
                # đi qua đúng đường `observe()` của bản chạy thật, không dựng lại phần
                # đếm phiếu ở đây — dựng lại là tự tạo ra một bản sao để lệch.
                tracker.sticky.clear()

            row = {"device": device, "window_id": w["window_id"],
                   "n_records": len(w["records"])}
            for head in predictor.heads:
                r = out[head]
                row[f"{head}_status"] = r["status"]
                row[f"{head}_top1"] = r["top1"]
                row[f"{head}_conf"] = r["confidence"]
                row[f"{head}_fp"] = r["fp"]
                row[f"{head}_via"] = r["retrieval_mode"]
                row[f"{head}_n_sources"] = r["n_sources"]
                row[f"{head}_threshold"] = r["threshold_used"]
                row[f"{head}_top2"] = r["top2"]
                row[f"{head}_margin"] = r["margin"]
                row[f"{head}_l1_status"] = r["l1_status"]
                row[f"{head}_hierarchy_status"] = r["hierarchy_status"]
                row[f"{head}_decision_reason"] = r["decision_reason"]
                row[f"{head}_policy_violation"] = int(r["policy_violation"])
                row[f"{head}_source"] = r.get("source")
                row[f"{head}_open_vocabulary"] = int(bool(r.get("open_vocabulary")))
                row[f"{head}_semantic_score"] = r.get("semantic_score")
                row[f"{head}_catalog_version"] = r.get("catalog_version")
                row[f"{head}_semantic_evidence"] = json.dumps(
                    r.get("semantic_evidence", []), ensure_ascii=False
                )
            window_rows.append(row)

        status = tracker.status()
        base = {"device": device,
                "n_window": status["windows"],
                "enroll_mode": status["enroll_mode"] or "",
                "no_ip_stack": meta.get("no_ip_stack", 0)}
        for head in predictor.heads:
            s = status[head]
            gt = truth_for(meta, head)
            in_train = bool(gt) and gt in classes[head]
            base[f"{head}_truth"] = gt
            base[f"{head}_in_train"] = int(in_train)
            base[f"{head}_state"] = s["state"]
            base[f"{head}_top1"] = s["top1"] or ""
            base[f"{head}_answer_ratio"] = s["ratios"]["answer"]
            compatible = bool(s["top1"] == gt)
            if (head == "type" and predictor.semantic_type is not None
                    and predictor.semantic_type.is_compatible(s["top1"], gt)):
                compatible = True
            # Chỉ coi là "đã trả lời" khi tracker thật sự kết luận. `top1` lúc state còn
            # `collecting`/`unstable` mới là phiếu dẫn đầu, chưa phải câu trả lời.
            pred = s["top1"] if s["state"] == "identified" else ""
            base[f"{head}_pred"] = pred or ""
            base[f"{head}_match"] = match_mark(pred, gt, compatible)
            base[f"{head}_verdict"] = judge(
                s["state"], s["top1"], gt, in_train, compatible
            )
        device_rows.append(base)

    windows_df = pd.DataFrame(window_rows)
    devices_df = pd.DataFrame(device_rows)

    # --- Báo cáo mức cửa sổ -----------------------------------------------------------
    print("=" * 100)
    print("MỨC CỬA SỔ — một lần thu thập nói được gì")
    print("=" * 100)
    for head in predictor.heads:
        st = windows_df[f"{head}_status"].value_counts(normalize=True)
        fp = windows_df[f"{head}_fp"].value_counts(normalize=True)
        l1 = windows_df[f"{head}_via"].isin(
            ["fp_full", "fp_dhcp", "fp_tls"]
        ).mean()
        semantic = windows_df[f"{head}_via"].eq("semantic_catalog").mean()
        print(f"  {head:7s} trả lời {st.get('answer', 0)*100:5.1f}%   "
              f"abstain {st.get('abstain', 0)*100:5.1f}%   "
              f"L1 trả lời {l1*100:5.1f}%   semantic {semantic*100:5.1f}%   "
              f"vân tay: hit {fp.get('hit', 0)*100:4.1f}% / "
              f"ambiguous {fp.get('ambiguous', 0)*100:4.1f}% / "
              f"miss {fp.get('miss', 0)*100:4.1f}%")

    # --- Báo cáo mức thiết bị ---------------------------------------------------------
    # Mỗi thiết bị một dòng: model nói gì ở make/type/family, đặt cạnh nhãn thật của bộ
    # IoT Sentinel. Ô trống = model không trả lời; đó là câu trả lời hợp lệ ở bài đo này
    # chứ không phải thiếu dữ liệu, nên để trống chứ không điền tên trạng thái nội bộ.
    W = 18                                   # bề rộng mỗi cột nhãn
    print()
    print("=" * 120)
    print("MỨC THIẾT BỊ — dự đoán  vs  nhãn thật (gộp mọi cửa sổ của cùng một MAC)")
    print("=" * 120)
    header = f"{'thiết bị':<20s} {'cs':>3s} "
    for head in predictor.heads:
        header += f"| {head.upper():^{2 * W + 3}s} "
    print(header)
    sub = f"{'':<20s} {'':>3s} "
    for _ in predictor.heads:
        sub += f"| {'dự đoán':<{W}s} {'nhãn thật':<{W}s}   "
    print(sub)
    print("-" * 120)
    for _, r in devices_df.sort_values("device").iterrows():
        star = "*" if r["no_ip_stack"] else " "
        line = f"{r['device']:<19s}{star} {r['n_window']:>3d} "
        for head in predictor.heads:
            pred = str(r[f"{head}_pred"]) or "·"
            gt = str(r[f"{head}_truth"]) or "·"
            line += f"| {pred[:W]:<{W}s} {gt[:W]:<{W}s} {r[f'{head}_match']} "
        print(line.rstrip())
    print()
    for mark, note in MARKS.items():
        if mark.strip():
            print(f"  [{mark}]  {note}")
    print("   ·   ô trống: model không trả lời, hoặc bộ test không có nhãn thật cho head đó")
    if devices_df.no_ip_stack.any():
        print("  * thiết bị ZigBee/Z-Wave thuần: traffic trong pcap là của gateway, "
              "không phải của chính nó")

    # --- Tổng kết ---------------------------------------------------------------------
    print()
    print("=" * 100)
    print("TỔNG KẾT")
    print("=" * 100)
    summary = {"run": str(predictor.run_dir), "n_device": len(devices_df),
               "n_window": len(windows_df), "sticky": not args.no_sticky,
               "contract_format": predictor.contract_format,
               "semantic_catalog_version": (predictor.semantic_type.catalog_version
                                            if predictor.semantic_type else None),
               "thresholds": predictor.thresholds, "heads": {}}

    for head in predictor.heads:
        counts = Counter(devices_df[f"{head}_verdict"])
        decided = devices_df[devices_df[f"{head}_verdict"] != "CHƯA ĐỦ"]
        n_known = int(decided[f"{head}_in_train"].sum())
        n_new = len(decided) - n_known
        n_skip = len(devices_df) - len(decided)
        print(f"\n  {head.upper()}  —  {n_known} thiết bị thuộc lớp đã biết, "
              f"{n_new} thiết bị lạ"
              + (f", {n_skip} chưa đủ cửa sổ" if n_skip else ""))
        for verdict, note in VERDICTS.items():
            n = counts.get(verdict, 0)
            if not n:
                continue
            base = {"OK": n_known, "SAI": n_known, "BỎ SÓT": n_known,
                    "CHƯA ĐỦ": len(devices_df)}.get(verdict, n_new)
            print(f"    {verdict:<14s} {n:3d}/{base:<3d}  {note}")
        summary["heads"][head] = {
            "n_known": n_known, "n_new": n_new, "n_undecided": n_skip,
            "verdicts": dict(counts),
            "policy_violation": int(windows_df[f"{head}_policy_violation"].sum()),
            # Hai con số đáng nhìn nhất, tách ra sẵn để so giữa các lần chạy.
            "ood_detection": round(counts.get("PHÁT HIỆN LẠ", 0) / n_new, 4) if n_new else None,
            "silent_error": round(counts.get("BÁO NHẦM", 0) / n_new, 4) if n_new else None,
            "open_identification": round(counts.get("OK MỞ", 0) / n_new, 4) if n_new else None,
            "known_accuracy": round(counts.get("OK", 0) / n_known, 4) if n_known else None,
        }

    summary["policy_violation"] = sum(
        values["policy_violation"] for values in summary["heads"].values()
    )

    print("\n  Chỉ số then chốt (thiết bị lạ bị gán nhãn = lỗi im lặng):")
    for head in predictor.heads:
        s = summary["heads"][head]
        fmt = lambda v: "  n/a " if v is None else f"{v*100:5.1f}%"
        print(f"    {head:7s} phát hiện lạ {fmt(s['ood_detection'])}   "
              f"lỗi im lặng {fmt(s['silent_error'])}   "
              f"nhận loại mở {fmt(s['open_identification'])}   "
              f"đúng trên lớp đã biết {fmt(s['known_accuracy'])}")

    # --- Ghi file ---------------------------------------------------------------------
    args.out.mkdir(parents=True, exist_ok=True)
    windows_df.to_csv(args.out / "window_predictions.csv", index=False, encoding="utf-8-sig")
    devices_df.to_csv(args.out / "device_report.csv", index=False, encoding="utf-8-sig")
    (args.out / "summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nĐã ghi:\n  {args.out / 'window_predictions.csv'}"
          f"\n  {args.out / 'device_report.csv'}\n  {args.out / 'summary.json'}")
    if summary["policy_violation"]:
        raise RuntimeError(
            f"Detected {summary['policy_violation']} model-policy violations"
        )
    return args.out

## Chuẩn bị cửa sổ IoT Sentinel từ pcap

Nếu chưa có `windows.jsonl` và `coverage.csv`, cell chạy bài đo sẽ tạo chúng ngay tại đây bằng extractor trong `03`. Đặt `REBUILD_SENTINEL=True` để dựng lại. Có thể dùng `SENTINEL_LIMIT` để thử nhanh rồi đặt lại `0` cho báo cáo đầy đủ.


In [4]:
# Nhãn đối chứng của IoT Sentinel (chỉ dùng để chấm điểm, không dùng để train).
SENTINEL_TRUTH = [('Aria', 'Fitbit', 'Weight Scale', ''),
 ('D-LinkCam', 'D-Link', 'IP Camera', ''),
 ('D-LinkDayCam', 'D-Link', 'IP Camera', ''),
 ('D-LinkDoorSensor', 'D-Link', 'Door Sensor', ''),
 ('D-LinkHomeHub', 'D-Link', 'Smart Hub', ''),
 ('D-LinkSensor', 'D-Link', 'Motion Sensor', ''),
 ('D-LinkSiren', 'D-Link', 'Siren', ''),
 ('D-LinkSwitch', 'D-Link', 'Smart Plug', ''),
 ('D-LinkWaterSensor', 'D-Link', 'Water Sensor', ''),
 ('EdimaxCam1', 'Edimax', 'IP Camera', ''),
 ('EdimaxCam2', 'Edimax', 'IP Camera', ''),
 ('EdimaxPlug1101W', 'Edimax', 'Smart Plug', ''),
 ('EdimaxPlug2101W', 'Edimax', 'Smart Plug', ''),
 ('EdnetCam1', 'Ednet', 'IP Camera', ''),
 ('EdnetCam2', 'Ednet', 'IP Camera', ''),
 ('EdnetGateway', 'Ednet', 'Smart Hub', ''),
 ('HomeMaticPlug', 'eQ-3', 'Smart Plug', ''),
 ('HueBridge', 'Philips', 'Smart Hub', 'Philips Hue Bridge'),
 ('HueSwitch', 'Philips', 'Remote', ''),
 ('iKettle2', 'Smarter', 'Kettle', ''),
 ('Lightify', 'Osram', 'Smart Hub', ''),
 ('MAXGateway', 'eQ-3', 'Smart Hub', ''),
 ('SmarterCoffee', 'Smarter', 'Coffee Maker', ''),
 ('TP-LinkPlugHS100', 'TP-Link', 'Smart Plug', ''),
 ('TP-LinkPlugHS110', 'TP-Link', 'Smart Plug', ''),
 ('WeMoInsightSwitch', 'Belkin', 'Smart Plug', ''),
 ('WeMoInsightSwitch2', 'Belkin', 'Smart Plug', ''),
 ('WeMoLink', 'Belkin', 'Smart Hub', ''),
 ('WeMoSwitch', 'Belkin', 'Smart Plug', ''),
 ('WeMoSwitch2', 'Belkin', 'Smart Plug', ''),
 ('Withings', 'Withings', 'Weight Scale', '')]
NO_IP_STACK = {'HueSwitch', 'MAXGateway', 'Lightify'}
SENTINEL_CAPTURES = (ROOT / "test_model" / "data_test" / "captures_IoT_Sentinel"
                     / "captures_IoT-Sentinel")
REBUILD_SENTINEL = False
SENTINEL_LIMIT = 0       # 0 = tất cả pcap; đặt N > 0 để thử nhanh
FILTER_SENTINEL_MAC = True


def read_sentinel_mac(device_dir):
    """MAC thật trong metadata IoT Sentinel, nếu có."""
    import re
    path = device_dir / "_iotdevice-mac.txt"
    if not path.is_file():
        return None
    raw = re.sub(r"[^0-9a-f]", "", path.read_text(encoding="utf-8", errors="replace").lower())[:12]
    if len(raw) != 12:
        return None
    return ":".join(raw[i:i + 2] for i in range(0, 12, 2))


def prepare_sentinel_data(captures=SENTINEL_CAPTURES, out=SENTINEL_OUT,
                          limit=0, filter_mac=True):
    """Đọc pcap thành windows.jsonl và coverage.csv bằng cùng extractor lúc phục vụ."""
    captures, out = Path(captures), Path(out)
    if not captures.is_dir():
        raise FileNotFoundError(f"Không thấy thư mục pcap IoT Sentinel: {captures}")
    out.mkdir(parents=True, exist_ok=True)
    truth = {device: (make, kind, model) for device, make, kind, model in SENTINEL_TRUTH}
    device_dirs = sorted(path for path in captures.iterdir() if path.is_dir())
    if not device_dirs:
        raise RuntimeError(f"Không có thư mục thiết bị trong {captures}")
    windows_path = out / "windows.jsonl"
    temporary = out / "windows.jsonl.tmp"
    coverage = []
    n_windows = n_records = 0
    try:
        with temporary.open("w", encoding="utf-8") as fh:
            for device_dir in device_dirs:
                device = device_dir.name
                mac = read_sentinel_mac(device_dir) if filter_mac else None
                pcaps = sorted(p for p in device_dir.glob("*.pcap") if not p.name.startswith("._"))
                if limit:
                    pcaps = pcaps[:limit]
                total = Counter()
                kept = 0
                for pcap_path in pcaps:
                    records, counts = read_pcap(pcap_path, device_mac=mac)
                    total.update(counts)
                    if not records:
                        continue
                    kept += 1
                    n_records += len(records)
                    fh.write(json.dumps({
                        "device": device, "mac": mac, "capture_file": pcap_path.name,
                        "window_id": f"{device}_{pcap_path.stem}", "records": records,
                    }, ensure_ascii=False) + "\n")
                n_windows += kept
                make, kind, model = truth.get(device, ("", "", ""))
                coverage.append({
                    "device": device, "mac": mac or "", "make": make, "type": kind,
                    "model": model, "no_ip_stack": int(device in NO_IP_STACK),
                    "n_pcap": len(pcaps), "n_window": kept,
                    "n_dhcp": total["dhcp"], "n_dns": total["dns"],
                    "n_mdns": total["mdns"], "n_tls": total["tls"],
                })
        pd.DataFrame(coverage).to_csv(out / "coverage.csv", index=False, encoding="utf-8")
        temporary.replace(windows_path)
    finally:
        temporary.unlink(missing_ok=True)
    print(f"IoT Sentinel: {len(coverage)} thiết bị, {n_windows} cửa sổ, {n_records} bản ghi")
    return windows_path, out / "coverage.csv"


## Chạy

In [5]:
# Chọn model: ưu tiên run_dir do 03/04/05 truyền vào; nếu mở riêng thì dùng run ghim
# hoặc candidate duy nhất trong Models/.
MODEL_RUN = "20260915_144403_verified_tiered"  # ví dụ: "20260913_162157_verified_tiered"
RUN_DIR = select_run_dir(MODEL_RUN or globals().get("run_dir"))
PER_DEVICE = 10
SEED = 0

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module=r"sklearn\.utils\.parallel")

predictor = Predictor(RUN_DIR, enrolled={})
print("Model  :", predictor.run_dir)
print("Head   :", predictor.heads)
print("Ngưỡng :", describe_policy(predictor))


Model  : /home/ubuntu/sepcung/02.SDC/Models/20260915_144403_verified_tiered
Head   : ['make', 'type', 'model']
Ngưỡng : make[1.00 0.98 0.96 0.52] min=2  type[0.99 0.96 0.79 0.80] min=2  model[1.00 0.94 0.80 0.57] min=2


In [6]:
# --- Bài 1: IoT Sentinel (thiết bị lạ) ---------------------------------------------
sentinel_out = SENTINEL_HOME / "data_test" / f"out_nb_{RUN_DIR.name}"
sentinel_data = SENTINEL_OUT / "windows.jsonl"
sentinel_coverage = SENTINEL_OUT / "coverage.csv"

if REBUILD_SENTINEL or not sentinel_data.is_file() or not sentinel_coverage.is_file():
    sentinel_data, sentinel_coverage = prepare_sentinel_data(
        limit=SENTINEL_LIMIT, filter_mac=FILTER_SENTINEL_MAC
    )
run_sentinel_test(["--run", str(RUN_DIR), "--data", str(sentinel_data),
                   "--coverage", str(sentinel_coverage), "--out", str(sentinel_out)])
sentinel_summary = json.loads((sentinel_out / "summary.json").read_text(encoding="utf-8"))
display(pd.DataFrame(sentinel_summary["heads"]).T)
assert sentinel_summary["policy_violation"] == 0, "model trả lời trái chính sách ngưỡng"


Model  : /home/ubuntu/sepcung/02.SDC/Models/20260915_144403_verified_tiered  (sdc-tiered-v2)
Ngưỡng : make[1.00 0.98 0.96 0.52] min=2  type[0.99 0.96 0.79 0.80] min=2  model[1.00 0.94 0.80 0.57] min=2
         ngưỡng xếp theo số nguồn bằng chứng 1..4, sàn L1 = 0.72
Lớp    : make=26  type=15  model=37
Catalog: 2026-09-12.1
Dữ liệu: 29 thiết bị, 510 cửa sổ



MỨC CỬA SỔ — một lần thu thập nói được gì
  make    trả lời   0.0%   abstain 100.0%   L1 trả lời   0.0%   semantic   0.0%   vân tay: hit  0.0% / ambiguous  0.0% / miss 76.5%
  type    trả lời  48.0%   abstain  52.0%   L1 trả lời   0.0%   semantic  48.0%   vân tay: hit  0.0% / ambiguous  0.0% / miss 76.5%
  model   trả lời   0.0%   abstain 100.0%   L1 trả lời   0.0%   semantic   0.0%   vân tay: hit  0.0% / ambiguous  0.0% / miss 76.5%

MỨC THIẾT BỊ — dự đoán  vs  nhãn thật (gộp mọi cửa sổ của cùng một MAC)
thiết bị              cs |                  MAKE                   |                  TYPE                   |                  MODEL                  
                         | dự đoán            nhãn thật            | dự đoán            nhãn thật            | dự đoán            nhãn thật            
------------------------------------------------------------------------------------------------------------------------
Aria                  20 | ·                  Fitbit            

,n_known,n_new,n_undecided,verdicts,policy_violation,ood_detection,silent_error,open_identification,known_accuracy
make,9,20,0,"{'PHÁT HIỆN LẠ': 20, 'BỎ SÓT': 9}",0,1.0,0.0,0.0,0.0
type,22,7,0,"{'OK MỞ': 4, 'OK': 10, 'PHÁT HIỆN LẠ': 3, 'BỎ ...",0,0.4286,0.0,0.5714,0.4545
model,1,28,0,"{'PHÁT HIỆN LẠ': 28, 'BỎ SÓT': 1}",0,1.0,0.0,0.0,0.0


In [7]:
# --- Bài 2: CIC 2022 (thiết bị đã biết) ---------------------------------------------
# CIC không còn pcap thô trên đĩa, chỉ còn bảng phiên đã qua aggregate() ở bước 02, nên
# đưa thẳng dòng feature vào tracker. Khác DeviceTracker đúng một chỗ: bỏ bước aggregate.
# Ghim bằng chứng, đếm phiếu, luật min_windows/decide_ratio đều của lớp cha.
class RowTracker(DeviceTracker):
    def observe(self, row):
        row = self._apply_sticky(dict(row))
        self.last_row = row
        out = self.p.predict_row(row)
        self.windows += 1
        for head in self.p.heads:
            result = out[head]
            self.states[head][bucket(result)] += 1
            if result["status"] == "answer":
                self.votes[head][result["top1"]] += 1
        return out


cic = normalize_heads(pd.read_parquet(SESSIONS_PATH))
picked = []
for _, group in cic.groupby("canonical_device", sort=False):
    picked.extend(group.sample(min(len(group), PER_DEVICE), random_state=SEED).index)
sample = cic.loc[picked]

rows = []
for device, group in sample.groupby("canonical_device", sort=True):
    tracker = RowTracker(predictor, min_windows=3, decide_ratio=0.7)
    for _, session in group.iterrows():
        tracker.observe(session)
    status = tracker.status()
    entry = {"device": device, "n_phien": status["windows"]}
    for head in predictor.heads:
        state = status[head]
        # `top1` lúc state còn collecting/unstable mới là phiếu dẫn đầu, chưa phải câu trả lời.
        pred = state["top1"] if state["state"] == "identified" else ""
        truth = truth_for(session, head)
        entry[f"{head}_pred"] = pred
        entry[f"{head}_truth"] = truth
        entry[f"{head}_verdict"] = "OK" if pred == truth else ("SAI" if pred else "BỎ SÓT")
    rows.append(entry)

cic_report = pd.DataFrame(rows)
cic_summary = pd.DataFrame({
    head: cic_report[f"{head}_verdict"].value_counts() for head in predictor.heads
}).T.fillna(0).astype(int)
display(cic_summary)
display(cic_report)

wrong = {head: cic_report.loc[cic_report[f"{head}_verdict"].eq("SAI"), "device"].tolist()
         for head in predictor.heads}
cic_doc = {
    "format": "sdc-cic-evaluation-v1", "run_id": RUN_DIR.name,
    "n_devices": len(cic_report), "n_sessions": int(cic_report.n_phien.sum()),
    "wrong_count": sum(len(items) for items in wrong.values()),
    "heads": {head: cic_report[f"{head}_verdict"].value_counts().to_dict()
              for head in predictor.heads},
}
sentinel_out.mkdir(parents=True, exist_ok=True)
cic_report.to_csv(sentinel_out / "cic_device_report.csv", index=False, encoding="utf-8-sig")
(sentinel_out / "cic_summary.json").write_text(
    json.dumps(cic_doc, indent=2, ensure_ascii=False), encoding="utf-8"
)
assert not any(wrong.values()), f"thiết bị bị gán sai nhãn: {wrong}"
print(f"{len(cic_report)} thiết bị, {int(cic_report.n_phien.sum())} phiên "
      f"— không nhãn nào bị trả lời sai")

,OK,BỎ SÓT
make,38,13
type,42,9
model,39,12


,device,n_phien,make_pred,make_truth,make_verdict,type_pred,type_truth,type_verdict,model_pred,model_truth,model_verdict
0,Amazon Alexa Echo Dot 1,10,Amazon,Amazon,OK,Smart Speaker,Smart Speaker,OK,Amazon Echo Dot,Amazon Echo Dot,OK
1,Amazon Alexa Echo Dot 2,10,Amazon,Amazon,OK,Smart Speaker,Smart Speaker,OK,Amazon Echo Dot,Amazon Echo Dot,OK
2,Amazon Alexa Echo Spot,10,Amazon,Amazon,OK,Smart Display,Smart Display,OK,Amazon Echo Spot,Amazon Echo Spot,OK
3,Amazon Alexa Echo Studio,10,Amazon,Amazon,OK,Smart Speaker,Smart Speaker,OK,Amazon Echo Studio,Amazon Echo Studio,OK
4,Amazon Plug,10,Amazon,Amazon,OK,Smart Plug,Smart Plug,OK,Amazon Smart Plug,Amazon Smart Plug,OK
5,Amcrest WiFi Camera,10,Amcrest,Amcrest,OK,IP Camera,IP Camera,OK,Amcrest WiFi Camera,Amcrest WiFi Camera,OK
6,Arlo Base Station,10,Arlo,Arlo,OK,Base Station,Base Station,OK,Arlo Base Station,Arlo Base Station,OK
7,Arlo Q Camera,10,Arlo,Arlo,OK,IP Camera,IP Camera,OK,Arlo Q,Arlo Q,OK
8,Atomi Coffee Maker,10,Tuya ODM,Tuya ODM,OK,Tuya Device,Tuya Device,OK,Tuya Plug,Tuya Plug,OK
9,Borun Sichuan-AI Camera,10,Borun,Borun,OK,IP Camera,IP Camera,OK,Borun Camera,Borun Camera,OK


51 thiết bị, 430 phiên — không nhãn nào bị trả lời sai


## Bộ đo FIELD capture



In [8]:
# --- Bài 3: FIELD capture (thiết bị nhà bạn, tự bắt gói) ---------------------------
CAPTURE_SCENARIO = "FIELD"

field_sessions = normalize_heads(pd.read_parquet(SESSIONS_PATH))
field_sessions = field_sessions[field_sessions["scenario"] == CAPTURE_SCENARIO]

if field_sessions.empty:
    print(f"Không có session nào scenario={CAPTURE_SCENARIO!r} trong {SESSIONS_PATH.name} "
          f"— chạy Code/02_2_add_capture_devices.ipynb rồi train lại trước khi đo bài này.")
    field_report = pd.DataFrame()
else:
    rows = []
    for device, group in field_sessions.groupby("canonical_device", sort=True):
        tracker = RowTracker(predictor, min_windows=3, decide_ratio=0.7)
        session = None
        for _, session in group.iterrows():
            tracker.observe(session)
        status = tracker.status()
        entry = {"device": device, "n_phien": status["windows"]}
        for head in predictor.heads:
            state = status[head]
            pred = state["top1"] if state["state"] == "identified" else ""
            truth = truth_for(session, head)
            entry[f"{head}_pred"] = pred
            entry[f"{head}_truth"] = truth
            entry[f"{head}_verdict"] = "OK" if pred == truth else ("SAI" if pred else "BỎ SÓT")
        rows.append(entry)

    field_report = pd.DataFrame(rows)
    field_summary = pd.DataFrame({
        head: field_report[f"{head}_verdict"].value_counts() for head in predictor.heads
    }).T.fillna(0).astype(int)
    display(field_summary)
    display(field_report)

    wrong = {head: field_report.loc[field_report[f"{head}_verdict"].eq("SAI"), "device"].tolist()
             for head in predictor.heads}
    field_doc = {
        "format": "sdc-field-evaluation-v1", "run_id": RUN_DIR.name,
        "n_devices": len(field_report), "n_sessions": int(field_report.n_phien.sum()),
        "wrong_count": sum(len(items) for items in wrong.values()),
        "heads": {head: field_report[f"{head}_verdict"].value_counts().to_dict()
                  for head in predictor.heads},
    }
    sentinel_out.mkdir(parents=True, exist_ok=True)
    field_report.to_csv(sentinel_out / "field_device_report.csv", index=False, encoding="utf-8-sig")
    (sentinel_out / "field_summary.json").write_text(
        json.dumps(field_doc, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    if any(wrong.values()):
        print(f"CẢNH BÁO — thiết bị field-capture bị gán sai nhãn (kiểm tra windowing/nhãn): {wrong}")
    print(f"{len(field_report)} thiết bị, {int(field_report.n_phien.sum())} phiên FIELD")


,BỎ SÓT,OK
make,10,2
type,5,7
model,10,2


,device,n_phien,make_pred,make_truth,make_verdict,type_pred,type_truth,type_verdict,model_pred,model_truth,model_verdict
0,Desktop PC DNGJHRT,5,,Generic Laptop,BỎ SÓT,Laptop,Laptop,OK,,Windows Desktop HP,BỎ SÓT
1,Desktop PC DucAnh,6,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop DELL,Windows Desktop DELL,OK
2,IP Camera (field),6,,Camera,BỎ SÓT,,IP Camera,BỎ SÓT,,Generic IP Camera,BỎ SÓT
3,Laptop VAF70SQ6,6,,Generic Laptop,BỎ SÓT,Laptop,Laptop,OK,,Windows Laptop HP,BỎ SÓT
4,Lee Kingdom Laptop,5,,Generic Laptop,BỎ SÓT,Laptop,Laptop,OK,,Windows Desktop DELL,BỎ SÓT
5,Linova Laptop (linova),6,,Linova/Linux,BỎ SÓT,Laptop,Laptop,OK,,Linova Laptop HP,BỎ SÓT
6,OPPO A92,1,,OPPO,BỎ SÓT,,Smartphone,BỎ SÓT,,OPPO A92,BỎ SÓT
7,Raspberry Pi,3,,Raspberry Pi,BỎ SÓT,,Single-board Computer,BỎ SÓT,,Raspberry Pi,BỎ SÓT
8,Samsung Phone (Duc Anh),6,,Samsung,BỎ SÓT,Smartphone,Smartphone,OK,,Samsung Galaxy,BỎ SÓT
9,Xiaomi Redmi Note 10,1,,Xiaomi,BỎ SÓT,,Smartphone,BỎ SÓT,,Redmi Note 10,BỎ SÓT


12 thiết bị, 50 phiên FIELD
